In [59]:
import pandas as pd
import glob, os
from pathlib import Path

In [60]:
path_my = r'Z:\Калинкина\Стоматология_Бодерко\ОСП\посещения_039'

In [ ]:
mo_oid_for_filtr = '1.2.643.5.1.13.13.12.2.72.7357'

In [61]:
files = glob.glob(path_my + "//*.xlsx")

In [62]:
svod = []

In [63]:
for file in files:
    date = pd.read_excel(file, skiprows=15, sheet_name='TDSheet')[['Unnamed: 20']]
    if date.iloc[0, 0] != "1 января 2025 г. - 30 сентября 2025 г.":
        print(file)


In [64]:
for file in files:
    data = pd.read_excel(file, skiprows=28, sheet_name='TDSheet')
    name = os.path.basename(file).split('.')[0]
    data = data.drop(columns=['Unnamed: 119'])
    data = data.dropna(axis=1, how='all')
    data = data.dropna(how='all')
    data = data[data['1'] != 'Итого']
    data.insert(0, 'mo_name', name)

    svod.append(data)

In [65]:
df = pd.concat(svod)

In [66]:
df.rename(columns={'mo_name' : 'Наименование МО',
                   '1' : 'Сотрудник',
                   '2' : 'Число посещений в поликлинике всего',
                   '3' : "из них сельских жителей",
                   '4' :  "В том числе 0-17 лет",
                   '5' : "В том числе 60 лет и старше",
                '6' : "Из общего числа посещений в поликлинике по поводу заболеваний всего",
                '7' : "в том числе в возрасте 0-17 лет",
                '8' : "в том числе в возрасте 60 лет и старше",
                '9' : "Профилактических",
                '10' : "Число посещений на дому всего",
                '11' : "Из общего числа посещений на дому по поводу заболеваний всего",
                '12' : "в том числе в возрасте 0-17 лет",
                '13' : "в том числе в возрасте 0-1 год вкл",
                '14' : "в том числе в возрасте 60 лет и старше",
                '15' : "Из общего числа посещений на дому из числа профилактических 0 - 17 лет",
                '16' : "Из общего числа посещений на дому из числа профилактических в т.ч. 0-1 год",
                '17' : "ОМС",
                '18' : "бюджет",
                '19' : "платные",
                '20' : "ДМС"}, inplace=True)

In [67]:
df['doctors'] = df['Сотрудник'].str.title()
df['doctors'] = df['doctors'].str.replace(" ", "_")
translation_table = str.maketrans("ё", "е")
df['doctors'] = df['doctors'].apply(
    lambda x: x.translate(translation_table) if isinstance(x, str) else x
)

In [ ]:
workers_2 = pd.read_excel('H:\\СЦ\\apache\\значение из сводов ЕГИС 2024\\workers_1.xlsx', sheet_name='frmr_depart', skiprows=6)
workers_2.columns = workers_2.columns.str.strip()
if workers_2['Unnamed: 0'].isna().all():
    workers_2 = workers_2.drop(['Unnamed: 0'], axis=1)
    workers_2.columns = [f'Unnamed: {int(col.split(": ")[-1]) - 1}' if col.startswith('Unnamed') else col for col in workers_2.columns]
workers_2.head()


# открываем workers и переименуем столбцы
workers_2 = workers_2[['OID организации',
    'OID организации',
    'Unnamed: 1',
    'Unnamed: 2',
    'Unnamed: 3',
    'Unnamed: 4',
    'Unnamed: 5',
    'Должность по фед справочнику']]
workers_2.columns = ['mo_oid', 'depart_oid','last_name', 'name', 'middle_name', 'sex','bday', 'job']
workers_2.dropna(subset=['job'], inplace=True) #удалим строки, где нет должности
workers_2['doctors'] = workers_2['last_name'] +' ' + workers_2['name'] +' ' + workers_2['middle_name'] #соединяем ФИО
workers_2['doctors'] = workers_2['doctors'].str.title()
workers_2['doctors'] = workers_2['doctors'].str.replace(" ", "_")
translation_table = str.maketrans("ё", "е")
workers_2['doctors'] = workers_2['doctors'].apply(
    lambda x: x.translate(translation_table) if isinstance(x, str) else x
)

workers_2['job'] = workers_2['job'].str.lower().str.strip() #все должности прописываем прописными буквами
workers_2 = workers_2[workers_2['mo_oid'] == mo_oid_for_filtr][['job', 'doctors']]
workers_2 = workers_2.drop_duplicates()

c:\Users\koreshilova_ea\Desktop\koreshok\doctors\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [69]:
df_work = df.merge(workers_2, how='left', on='doctors')

In [70]:
df_work

,Наименование МО,Сотрудник,Число посещений в поликлинике всего,из них сельских жителей,В том числе 0-17 лет,В том числе 60 лет и старше,Из общего числа посещений в поликлинике по поводу заболеваний всего,в том числе в возрасте 0-17 лет,в том числе в возрасте 60 лет и старше,Профилактических,...,в том числе в возрасте 0-1 год вкл,в том числе в возрасте 60 лет и старше,Из общего числа посещений на дому из числа профилактических 0 - 17 лет,Из общего числа посещений на дому из числа профилактических в т.ч. 0-1 год,ОМС,бюджет,платные,ДМС,doctors,job
0,ОСП_9мес_посещения,Аванесова Виктория Евгеньевна,1554.0,161.0,0.0,210.0,668.0,0.0,104.0,886.0,...,0.0,0.0,0.0,0.0,1292.0,0.0,261.0,0.0,Аванесова_Виктория_Евгеньевна,врач-стоматолог
1,ОСП_9мес_посещения,Алиева Айнура Алиевна,1505.0,147.0,0.0,223.0,511.0,0.0,84.0,994.0,...,0.0,0.0,0.0,0.0,1159.0,0.0,335.0,0.0,Алиева_Айнура_Алиевна,врач-стоматолог-терапевт
2,ОСП_9мес_посещения,Ардуванова Алина Ринатовна,1380.0,75.0,0.0,210.0,252.0,0.0,50.0,1128.0,...,0.0,0.0,0.0,0.0,984.0,0.0,394.0,0.0,Ардуванова_Алина_Ринатовна,врач-стоматолог
3,ОСП_9мес_посещения,Арутюнян Лусине Гагиковна,4390.0,473.0,7.0,1055.0,647.0,7.0,133.0,3743.0,...,0.0,0.0,0.0,0.0,3779.0,0.0,605.0,0.0,Арутюнян_Лусине_Гагиковна,зубной врач
4,ОСП_9мес_посещения,Асланов Ражидин Шарафудинович,3191.0,431.0,0.0,1140.0,155.0,0.0,68.0,3036.0,...,0.0,0.0,0.0,0.0,2890.0,0.0,301.0,0.0,Асланов_Ражидин_Шарафудинович,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,ОСП_9мес_посещения,Шипин Андрей Алексеевич,1310.0,104.0,0.0,1201.0,1304.0,0.0,1195.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1310.0,0.0,Шипин_Андрей_Алексеевич,врач-стоматолог-ортопед
161,ОСП_9мес_посещения,Шипин Никита Андреевич,1030.0,102.0,0.0,887.0,1030.0,0.0,887.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1030.0,0.0,Шипин_Никита_Андреевич,врач-стоматолог-ортопед
162,ОСП_9мес_посещения,Шнайдер Екатерина Сергеевна,3531.0,299.0,0.0,1233.0,1411.0,0.0,542.0,2120.0,...,0.0,0.0,0.0,0.0,3428.0,0.0,103.0,0.0,Шнайдер_Екатерина_Сергеевна,врач-стоматолог-хирург
163,ОСП_9мес_посещения,Шураева Алина Арсланалиевна,1485.0,104.0,0.0,282.0,452.0,0.0,111.0,1033.0,...,0.0,0.0,0.0,0.0,1304.0,0.0,179.0,0.0,Шураева_Алина_Арсланалиевна,врач-стоматолог


In [71]:
# текущие имена столбцов
cols = list(df_work.columns)

# убираем 'работник' из списка
cols.remove('job')

# вставляем 'работник' на индекс 2
cols.insert(2, 'job')

# применяем новый порядок столбцов
df_work = df_work[cols]


In [72]:
df_work.drop(columns=['doctors'], inplace=True)

C:\Users\koreshilova_ea\AppData\Local\Temp\ipykernel_12640\318909585.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_work.drop(columns=['doctors'], inplace=True)


In [73]:
df_work

,Наименование МО,Сотрудник,job,Число посещений в поликлинике всего,из них сельских жителей,В том числе 0-17 лет,В том числе 60 лет и старше,Из общего числа посещений в поликлинике по поводу заболеваний всего,в том числе в возрасте 0-17 лет,в том числе в возрасте 0-17 лет,...,в том числе в возрасте 0-17 лет,в том числе в возрасте 0-1 год вкл,в том числе в возрасте 60 лет и старше,в том числе в возрасте 60 лет и старше,Из общего числа посещений на дому из числа профилактических 0 - 17 лет,Из общего числа посещений на дому из числа профилактических в т.ч. 0-1 год,ОМС,бюджет,платные,ДМС
0,ОСП_9мес_посещения,Аванесова Виктория Евгеньевна,врач-стоматолог,1554.0,161.0,0.0,210.0,668.0,0.0,0.0,...,0.0,0.0,104.0,0.0,0.0,0.0,1292.0,0.0,261.0,0.0
1,ОСП_9мес_посещения,Алиева Айнура Алиевна,врач-стоматолог-терапевт,1505.0,147.0,0.0,223.0,511.0,0.0,0.0,...,0.0,0.0,84.0,0.0,0.0,0.0,1159.0,0.0,335.0,0.0
2,ОСП_9мес_посещения,Ардуванова Алина Ринатовна,врач-стоматолог,1380.0,75.0,0.0,210.0,252.0,0.0,0.0,...,0.0,0.0,50.0,0.0,0.0,0.0,984.0,0.0,394.0,0.0
3,ОСП_9мес_посещения,Арутюнян Лусине Гагиковна,зубной врач,4390.0,473.0,7.0,1055.0,647.0,7.0,0.0,...,0.0,0.0,133.0,0.0,0.0,0.0,3779.0,0.0,605.0,0.0
4,ОСП_9мес_посещения,Асланов Ражидин Шарафудинович,NaN,3191.0,431.0,0.0,1140.0,155.0,0.0,0.0,...,0.0,0.0,68.0,0.0,0.0,0.0,2890.0,0.0,301.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,ОСП_9мес_посещения,Шипин Андрей Алексеевич,врач-стоматолог-ортопед,1310.0,104.0,0.0,1201.0,1304.0,0.0,0.0,...,0.0,0.0,1195.0,0.0,0.0,0.0,0.0,0.0,1310.0,0.0
161,ОСП_9мес_посещения,Шипин Никита Андреевич,врач-стоматолог-ортопед,1030.0,102.0,0.0,887.0,1030.0,0.0,0.0,...,0.0,0.0,887.0,0.0,0.0,0.0,0.0,0.0,1030.0,0.0
162,ОСП_9мес_посещения,Шнайдер Екатерина Сергеевна,врач-стоматолог-хирург,3531.0,299.0,0.0,1233.0,1411.0,0.0,0.0,...,0.0,0.0,542.0,0.0,0.0,0.0,3428.0,0.0,103.0,0.0
163,ОСП_9мес_посещения,Шураева Алина Арсланалиевна,врач-стоматолог,1485.0,104.0,0.0,282.0,452.0,0.0,0.0,...,0.0,0.0,111.0,0.0,0.0,0.0,1304.0,0.0,179.0,0.0


In [74]:
Path(path_my + "\\result").mkdir(parents=True, exist_ok=True)

In [75]:
df_work.to_excel(path_my + '\\result\\result.xlsx', index=False)

PermissionError: [Errno 13] Permission denied: 'Z:\\Калинкина\\Стоматология_Бодерко\\ОСП\\посещения_039\\result\\result.xlsx'